# Beyond the Data Store Graphical User Interface (GUI): Hands-On with pyBIS
**Beginner friendly workshop | 210 minute**

## Introduction

### Welcome!

In the previous Lab Notebook tutorial, you learned how to organize and document research data using the BAM Data Store graphical user interface (GUI).

In this workshop, you will explore another way of interacting with the Data Store: **pyBIS**, a Python library that allows you to access and manage openBIS data through code.

**No previous experience with Python, Jupyter Notebooks, or pyBIS is required.** We will guide you through the exercises step by step.

### What will you learn?

By the end of this workshop, you will be able to:

1. **Retrieve research data:** Find and access your projects, collections, and experimental steps using Python.
2. **Modify metadata:** Update the description of a practice record and verify your changes in the Data Store GUI.
3. **Automate repetitive tasks:** Retrieve information from multiple records and export it as a table (CSV file).

### What data will we use?

This workshop builds on the [BAM Lab Notebook tutorial](https://bamresearch.github.io/openBIS-data-store-tutorials/lab-notebook/) and is adapted from `lab-notebook-pybis.ipynb`.

Rather than repeating the entire tutorial, we will work with a small example research workflow consisting of:

- A project called `Material_Synthesis`.
- Three collections: `Synthesis`, `Parameter 1`, and `Parameter 2`.
- A few experimental steps (`EXPERIMENTAL_STEP` objects) representing research activities.

You will either reuse data from the previous GUI tutorial or work with example data prepared for this workshop.

### Before you begin

- **Use only your assigned training Space.** This keeps your exercises separate from other participants' work.
- **Keep the original tutorial data unchanged.** We will create a dedicated practice record (`Dummy_Object`) for the metadata editing exercise.
- **Your computer should already be prepared.** VS Code, Python, pyBIS, and the required environment should have been installed following the separate setup instructions. No terminal commands or additional installation should be necessary during the workshop.

---

## Glossary — Understanding openBIS and pyBIS terminology

The BAM Data Store is based on **openBIS**. Some concepts have different names in the **Data Store graphical user interface (GUI)** and **pyBIS (Python interface)**.

This glossary explains the most important terms used throughout the workshop.

### Organization of research data

- **Space:** The highest organizational level in openBIS, used to organize research data and manage access permissions, for example, by separating working groups. You have permission to access the content of your assigned Lab Notebook Space, but you may not have access to other Spaces unless additional permissions are granted.

- **Project:** A Space can contain multiple Projects. Projects organize related research activities and their associated Collections.

- **Collection:** A Project can contain multiple Collections, Collections group related research records and can have metadata properties.

- **Object:** A Collection can contain multiple Objects. Objects represent research entities or activities, such as an experimental step, and can have metadata properties.

- **Dataset:** A Dataset contains the actual research data files. In the Data Store, Datasets are typically attached to Objects that describe how the data was generated, often an Object of type `EXPERIMENTAL_STEP`.

### Attributes and properties

- **Attributes:** System-defined fields that describe an openBIS entity, such as `code`, `permId`, `identifier`, `type`, and `registrationDate`.
  Attributes are available for common entity categories, including Spaces, Projects, Collections, Objects, and Datasets.

- **Properties:** Type-specific metadata fields used to describe research objects in a structured way.
  Properties are defined through *Property Types* and assigned to *Entity Types* during the masterdata definition process.

  Properties are supported for **Collections, Objects, and Datasets**. Spaces and Projects have system-defined attributes but do not contain type-specific properties.

  **Example:** An experimental step might have properties describing the temperature, duration, or responsible person.

### Entity types and property types

- **Entity Type:** Defines which metadata properties are available for a particular category of research record.

  In openBIS, Entity Types include:

  - **Collection Type:** Defines the properties available for a Collection.
  - **Object Type:** Defines the properties available for an Object.
  - **Dataset Type:** Defines the properties available for a Dataset.

  In the BAM Data Store, the Lab Notebook primarily uses predefined Collection and Dataset Types, while different Object Types represent specific research activities or entities and are user defined in within the frame of BAM Data Model.

- **Property Type:** Defines an individual metadata field that can be assigned to one or more Entity Types.
  Each Property Type has a data type that determines which values it can store, for example:

  - `INTEGER` — Whole numbers (e.g., 25)
  - `VARCHAR` — Text (e.g., "Steel")
  - `BOOLEAN` — True or false
  - `CONTROLLEDVOCABULARY` — A value selected from a predefined list

### Plugins
- **Plugin:** A script that extends openBIS functionality, for example, by performing additional checks on metadata values.
  Some openBIS plugins are written in [Jython](https://www.jython.org/), a Python implementation that runs on the Java platform.
  Plugins can help validate entered metadata and enforce specific rules.

---


## Research Data Hierarchy — pyBIS Workshop Example

In the BAM Data Store, research data is organized hierarchically:

**Space → Project → Collection → Object → Dataset**


The diagram below shows the entities we will use during the hands-on challenges (1–5).

```text
SPACE: YOUR_LAB_NOTEBOOK_SPACE
│
└── PROJECT: Material_Synthesis
    │
    ├── COLLECTION: Parameter_1
    │   │
    │   ├── OBJECT: Macrostructure_Characterization
    │   │   
    │   │
    │   └── OBJECT: Macrostructure_Characterization_TS
    │      
    │
    ├── COLLECTION: Parameter_2
    │   │
    │   └── OBJECT: Microstructure_Characterization
    │       
    │
    └── COLLECTION: Synthesis
        │
        ├── OBJECT: Material_Synthesis
        │   
        │
        └── OBJECT: Dummy_Object
            └── DATASET(S):  Zip file; of types RAW_DATA and ELN_PREVIEW
```

### Understanding the hierarchy

- **Space:** Your assigned Lab Notebook Space. It determines where you work and which research records you can access.
- **Project:** `Material_Synthesis` organizes the research activities used in this workshop.
- **Collections:** `Parameter_1`, `Parameter_2`, and `Synthesis` group related experimental activities.
- **Objects:** Represent individual research activities, such as material synthesis or characterization.
- **Datasets:** Contain the actual research data files associated with an Object.

### Important for the hands-on exercises

- We will **retrieve and explore** existing research records using pyBIS.
- We will use `Dummy_Object` as a **practice Object** for modifying metadata.
- We will **not modify the original research Objects** created during the GUI tutorial.
- We will **upload data** of different types and formats to the `Dummy_Object`
- we will **retrieve objects and export metadata** 

**Remember:** The Data Store GUI and pyBIS use different terminology, but they access the same research records.


## Step 0 - Check your workshop environment

Before we start working with the Data Store, let's make sure your Python environment is working correctly.

You should have already installed VS Code, Python, pyBIS, and the required libraries following the separate setup instructions.

**What to do:**

1. Open this notebook in VS Code
2. Make sure you have selected the Python environment prepared during setup <code>pybis(Python 3.14.8) or pybis(Python 3.14.10)</code>
3. Run the following code cell using the **▶ Run** button or **Shift + Enter**.
4. Check that the cell finishes without an error.

**Expected result**: You should see a confirmation message "Environment check passed" indicating that Python and the required libraries are available.


If you see an error, ask an instructor for help before continuing.


In [ ]:
import sys
from pathlib import Path

expected_env_name = "pybis"
active_prefix = Path(sys.prefix)

is_conda_env = (active_prefix / "conda-meta").is_dir()
is_expected_env = active_prefix.name.casefold() == expected_env_name.casefold()
is_python_314 = sys.version_info[:2] == (3, 14)

print("Python executable:", sys.executable)
print("Environment:", active_prefix)
print("Conda environment:", is_conda_env)
print("Expected environment (pybis):", is_expected_env)
print("Python version:", sys.version.split()[0])

if not (is_conda_env and is_expected_env and is_python_314):
    raise RuntimeError("Select the pybis Conda environment with Python 3.14, then rerun this cell.")

print("Environment check passed.")

In [ ]:
from importlib.metadata import PackageNotFoundError, version

print("Python executable:", sys.executable)

for package in ("pybis", "ipykernel"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError as error:
        raise RuntimeError(f"{package} is not installed in this environment.") from error

print("Both packages are installed and import successfully.")

In [ ]:
#commented libraries are listed in case some users experience issues
#you don't need to install them if you got the message "Environment check passed" in the previous cell.

# %pip install pybis
# %pip install ipykernel
%pip install pandas
# # %pip install jupyterlab


## Part A — Connect and prepare
### 1. Connect to openBIS
1. Open the BAM Data Store playground in your browser and log in.
2. In **Tools → User Profile**, copy your openBIS session token (as described in the source notebook).
3. Run the next cell with **Shift + Enter**. Paste the token into the hidden prompt. **Never paste a token into a saved code cell or share it.**

In [ ]:
from getpass import getpass
import pandas as pd
from pybis import Openbis
from helper import *

o = Openbis("https://playground.datastore.bam.de/")
o.set_token(getpass("Paste your openBIS session token (hidden): "))
print("pyBIS connection configured. Next, select your Space.")

## Step 1: Select your own Space
Replace the placeholder with your assigned Space code.

In [ ]:
display(o.get_spaces())# This will display a list of spaces you have access to. You can find your space code in the list and copy it.

In [31]:
myspace = o.get_space('VP.1_NARSHAD')# Paste your space code here.

## Step 2: Check that the Project, Collections, and experimental steps needed for workshop exercises are registered in your Lab Notebook

**Path A:** You completed the GUI tutorial. The notebook looks for the existing `Material_Synthesis` project and its named collections.

**Path B:** You did not complete the GUI tutorial. The notebook creates **only missing** minimal structures: a project, three collections and a few demonstration experimental steps.

Run the same cells for either path. **No need to decide manually.** The checks are based on actual objects.

In [32]:
PROJECT_CODE = "Material_Synthesis_1234567890"

object_codes = {
    'Macrostructure Characterization': 'Parameter 1',
    'Macrostructure Characterization TS': 'Parameter 1',
    'Microstructure Characterization': 'Parameter 2',
    'Material Synthesis': 'Synthesis'
}
collection_codes = ["Parameter 1", "Parameter 2", "Synthesis"]
project_missing_flag = False
missing_collections = []
missing_objects = []
path = 'B'#default value


try:#try except for catching no project error
    project = myspace.get_project(project_code=PROJECT_CODE)
except Exception:
    print('Project Material Synthesis Missing')
    project_missing_flag = True
    missing_collections = collection_codes.copy()

if not project_missing_flag:#if project exists
    project_collections = project.get_collections()
    collections_by_name = {
        collection.props.get('$name'): collection
        for collection in project_collections
    }
    missing_collections = [
        name for name in collection_codes if name not in collections_by_name
    ]
    for template_name, collection_name in object_codes.items():
        collection = collections_by_name.get(collection_name)
        if collection is None:
            continue
        existing_object_names = {
            object.props.get('$name') for object in collection.get_objects()
        }
        if template_name not in existing_object_names:
            missing_objects.append((template_name, collection_name))

if not project_missing_flag and not missing_collections and not missing_objects:
    print('Project, collections, and expected objects exist')
    path = 'A'

print('Missing collections:', missing_collections)
print('Missing objects:', missing_objects)
print('Path =', path)

Project Material Synthesis Missing
Missing collections: ['Parameter 1', 'Parameter 2', 'Synthesis']
Missing objects: []
Path = B


In [33]:
# If Path B, create only the entities that are missing.
if path == 'B':
    if project_missing_flag:
        project = myspace.new_project(
            code=PROJECT_CODE,
            description='This tutorial project demonstrates a simplified workflow for the synthesis and characterization of a material within openBIS. The experimental descriptions, metadata, and datasets were generated using AI and represent synthetic example data created solely for training and demonstration purposes. Although the content has been reviewed for consistency and usability within the tutorial, it has not been validated for scientific accuracy and should not be considered scientifically reliable or reproducible.'
        )
        project.save()
        collections_to_create = collection_codes
    else:
        collections_to_create = missing_collections

    for name in collections_to_create:
        collection = o.new_collection(
            code=auto_generated_code(project),
            type='Collection',
            project=project.identifier,
            props={'$name': name}
        )
        collection.save()

    collections_by_name = {
        collection.props.get('$name'): collection
        for collection in project.get_collections()
    }

    for template_name, collection_name in object_codes.items():
        collection = collections_by_name[collection_name]
        existing_object_names = {
            object.props.get('$name') for object in collection.get_objects()
        }
        if template_name in existing_object_names:
            continue

        if template_name == 'Macrostructure Characterization TS':
            object_props = templates['Macrostructure Characterization'].copy()
        else:
            object_props = templates[template_name].copy()
        object_props['$name'] = template_name

        template_object = o.new_object(
            type='EXPERIMENTAL_STEP',
            collection=collection,
            props=object_props
        )
        template_object.save()

project successfully created.
experiment successfully created.
experiment successfully created.
experiment successfully created.
sample successfully created.
sample successfully created.
sample successfully created.
sample successfully created.


**GUI checkpoint:** Open the Material Synthesis project. Can you find **Synthesis**, **Parameter 1**, **Parameter 2** and a total of four Experimental Steps called:
'Macrostructure Characterization', 'Macrostructure Characterization TS', 'Microstructure Characterization', and 'Material Synthesis'? If yes, you are ready for the hands-on exercises.

---
## Step 3: Hands-on challenges
### Challenge 1 — Retrieve a record: Find an experimental step (30 minutes)

In the Data Store web interface, you can navigate through projects and collections to find an experimental step.

We will now perform the same operation using Python.

**What the code does:**

1. Looks at the collections in your project.
2. Finds the collection with the requested name.
3. Retrieves the experimental steps stored in that collection.
4. Displays the name and identifier of one experimental step.

An **identifier** is a reference used by openBIS to distinguish one record from another.

**Your task:** Change the collection name from `Parameter 2` to `Synthesis` and run the cell again.

**Expected result:** You should see the name and identifier of an experimental step in the Synthesis collection.

**Check:** Can you find the same experimental step in the Data Store web interface?


In [34]:
# STEP 1: Display the identifier of the project we are exploring

print("Project:", project.identifier)

# STEP 2: Find the collection named "Parameter 2"

# Empty Variables to hold the collection and its objects
collection = None
objects = None

for collection in project.get_collections():
    # Check the name of each collection and match it with the name we are looking for

    if collection.props.all()["$name"] == 'Parameter 2':

        #Store the objects in the variable for the next step
        objects = collection.get_objects()
        break

# STEP 3: Display the selected collection and its first object
# The object name is the human-readable name; the identifier uniquely identifies it

print('Collection: ',collection.props.all()["$name"])
print('Object Name: ',objects[0].props.all()['$name'])
print('Object Identifier: ',objects[0].identifier)

Project: /VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890
Collection:  Parameter 2
Object Name:  Microstructure Characterization
Object Identifier:  /VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EXP8871


### Your task: replace the collection name from `Parameter 2` to `Synthesis`.

In [35]:
# Empty Variable to hold the objects
objects = None

for collection in project.get_collections():
    # Check the name of each collection and match it with the name we are looking for

    if collection.props.all()["$name"] == 'Parameter 2':# <-- Change the name to Synthesis
        #Store the objects in the variable for later use

        objects = collection.get_objects()
        break

# Display the object's name and identifier.

print('Object Name: ',objects[0].props.all()['$name'])
print('Object Identifier: ',objects[0].identifier)

Object Name:  Microstructure Characterization
Object Identifier:  /VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EXP8871


**Check:** Can you find the same experimental step in the Data Store web interface?

**Expected result:** You should see the name and identifier of an experimental step in the Synthesis collection.

### Challenge 2 — Explore your research data as a table (30 minutes)

When a project contains many experimental steps, inspecting them one by one can be time-consuming.

Python can retrieve information from multiple records and organize it into a table.

In this exercise, we use **pandas**, a Python library for working with tabular data.

**The table contains four columns:**

- `collection`: The collection containing the experimental step.
- `identifier`: The unique identifier of the record.
- `name`: The human-readable name of the experimental step.
- `type`: The openBIS object type.

You do not need to understand every line of code. Focus on the table produced by the code.

**Your task:** Change `collection_to_find` from `Parameter 1` to `Synthesis`.

**Expected result:** The displayed table contains only experimental steps from the selected collection.

**Think about it:** How could filtering help you find experimental records in a project containing hundreds of steps?


In [36]:
# STEP 1: Create an empty list to collect information about our objects
# Each object will later become one row in our table
rows = []

# STEP 2: Select the two collections we want to explore
# We look for collections named "Synthesis" and "Parameter 1"
# The next() function returns the first collection matching the name

for collection_name, collection in [
    ("Synthesis", next(col for col in project.get_collections()if col.props["$name"] == "Synthesis")),
    ("Parameter 1", next(col for col in project.get_collections()if col.props["$name"] == "Parameter 1"))
]:

    # STEP 3: Retrieve all objects in the selected collection

    for obj in collection.get_objects():

        # STEP 4: Extract information from each object
        # Store the information as a dictionary (key-value pairs)
        # Each dictionary will become one row in the table

        rows.append({
            "collection": collection_name,   # Collection name
            "identifier": obj.identifier,     # Unique object identifier
            "name": obj.props.get("$name"),   # Human-readable object name
            "type": obj.type,                 # Object type in openBIS
        })

# STEP 5: Convert the collected information into a pandas DataFrame
# A DataFrame is a table with rows and columns

objects_table = pd.DataFrame(rows,columns=["collection", "identifier", "name", "type"])

# STEP 6: Display the table in the notebook
display(objects_table)

# STEP 7: Count and display how many objects were found
print("Total objects:", len(objects_table))


,collection,identifier,name,type
0,Synthesis,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EX...,Material Synthesis,EXPERIMENTAL_STEP
1,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EX...,Macrostructure Characterization TS,EXPERIMENTAL_STEP
2,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EX...,Macrostructure Characterization,EXPERIMENTAL_STEP


Total objects: 3


### Your task: Change `collection_to_find` from `Parameter 1` to `Synthesis`.


In [39]:
collection_to_find = "Parameter 1"  # <-- change this

# Filter the table to show only the objects in the selected collection
filtered = objects_table[objects_table["collection"] == collection_to_find]

# Display the filtered table
display(filtered)

# How many Objects are registered within this collection?
print("Matching objects:", len(filtered))

,collection,identifier,name,type
1,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EX...,Macrostructure Characterization TS,EXPERIMENTAL_STEP
2,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EX...,Macrostructure Characterization,EXPERIMENTAL_STEP


Matching objects: 2


**Hint:** The text must match a collection name exactly. Try changing `collection_to_find` and running the cell again.

### Challenge 3 — Modify metadata of an experimental step

#### Why do this?

Research metadata describes how an experiment was performed, for example, the materials, equipment, or experimental conditions used.

Sometimes, this information needs to be corrected or updated. Instead of opening each record in the Data Store graphical user interface (GUI), you can use pyBIS to modify metadata directly from Python.

**Important:** Retrieving data with pyBIS does not change anything in the Data Store. However, saving a modified object updates the information stored in openBIS.

In this exercise you will update the description of a practice Object called `Dummy_Object` in the `Synthesis` Collection.

You will perform three steps:

1. **Inspect:** Retrieve the Object and display its current description.
2. **Prepare a change:** Enter a new description and preview the change without saving it.
3. **Save and verify:** Save the new description, retrieve the Object again, and verify the change in the Data Store GUI.

#### Your task

1. Run the first code cell to retrieve `Dummy_Object` and inspect its current description.
2. Change the value of `new_description` to a short description of your choice.
3. Run the preview cell to check the proposed change.
4. Change `SAVE_CHANGE` from `False` to `True` when you are ready to save.
5. Run the save cell.
6. Open the Data Store GUI and find `Dummy_Object` in the `Synthesis` Collection.
7. Check whether the description has been updated.

#### Expected result

- The notebook displays the original description.
- You can preview the new description before saving.
- After saving, the updated description is stored in openBIS.
- The same description is visible in the Data Store GUI.

#### Important notes

- **Only modify `Dummy_Object`.** Do not change the original research Objects created during the previous GUI tutorial.
- The `.save()` method sends your changes to openBIS.
- Changes are not saved to openBIS until `.save()` is executed.
- If you run the save cell again, you may overwrite the description with the same or a different value.

**Think about it:** Why is it useful to inspect the existing metadata before modifying it?

In [40]:
# STEP 1: Find the Synthesis collection in our project

collection = next(col for col in project.get_collections()if col.props["$name"] == "Synthesis")

# STEP 2: Look for our practice Object
practice_name = "Dummy_Object"

matching_objects = [
    obj for obj in collection.get_objects()
    if obj.props.get("$name") == practice_name
]

# STEP 3: Reuse the practice Object if it already exists
# This prevents creating a duplicate when the cell is run again

if matching_objects:
    object = matching_objects[0]
    print("Existing practice Object found.")

else:
    # Create a new practice Object in the Synthesis collection

    object = o.new_object(
        type="EXPERIMENTAL_STEP",
        collection=collection,
        props={"$name": practice_name}
    )

    # Register the new Object in openBIS
    object.save()
    print("New practice Object created.")

# STEP 4: Define the metadata property we want to edit
PROPERTY = "experimental_step.experimental_description"

# STEP 5: Display the current metadata
print("Object Name:", object.props.get("$name"))
print("Identifier:", object.identifier)
print("Current description:", object.props.get(PROPERTY))

sample successfully created.
New practice Object created.
Object Name: Dummy_Object
Identifier: /VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EXP8873
Current description: None


In [41]:
# STEP 1: Enter a new description for the practice Object
# YOUR TURN: Change the text between the quotation marks

new_description = "I updated this training record using pyBIS!"

# STEP 2: Preview the proposed change
# Nothing is saved to openBIS at this stage

print("Current description:", object.props.get(PROPERTY))
print("Proposed description:", new_description)
print("No changes have been saved yet.")

Current description: None
Proposed description: I updated this training record using pyBIS!
No changes have been saved yet.


In [ ]:
# STEP 1: Decide whether to save the change
# YOUR TURN: Change False to True when you are ready

SAVE_CHANGE = False  # <-- Change this to True to save the change

# STEP 2: Save the new description only if confirmed
if SAVE_CHANGE:

    # Update the property in the Python Object
    object.props[PROPERTY] = new_description

    # Send the change to openBIS
    object.save()

    print("Changes saved successfully!")

else:
    # The original description remains unchanged
    print("Preview only. No changes saved.")


sample successfully updated.
Changes saved successfully!


In [44]:
# STEP 1: Retrieve the Object again from openBIS
# This ensures we check the information stored on the server

verified_object = o.get_object(object.identifier)

# STEP 2: Display the description stored in openBIS
print("Object Name:", verified_object.props.get("$name"))
print("Identifier:", verified_object.identifier)
print("Description in openBIS:", verified_object.props.get(PROPERTY))

# STEP 3: Compare the stored description with our proposed change
if verified_object.props.get(PROPERTY) == new_description:
    print("Verification successful: Descriptions match!")
else:
    print("The stored description differs from the proposed change.")

Object Name: Dummy_Object
Identifier: /VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EXP8873
Description in openBIS: I updated this training record using pyBIS!
Verification successful: Descriptions match!


**GUI checkpoint:**
- Open the Data Store GUI, refresh the page, and find `Dummy_Object` in the `Synthesis` Collection.
- Check whether the description has been updated.

### Challenge 4 - Upload datasets to the Data Store using pyBIS  (30 minutes)

#### Why do this?

Research experiments often generate many data files, such as images, measurements, or documentation.

In the Data Store graphical user interface (GUI), you can upload these files manually and attach them to an experimental step.

With **pyBIS**, you can perform the same task using Python. This is particularly useful when you need to upload multiple files or automate repetitive data management tasks.

#### In this exercise...

You will upload three different types of datasets to the practice Object `Dummy_Object` in the `Synthesis` Collection.

You will perform four steps with pyBIS:

1. **Prepare the files:** Extract a ZIP archive containing example research data.
2. **Upload raw data:** Register the extracted files as a `RAW_DATA` dataset.
3. **Upload a preview:** Register an example image as an `ELN_PREVIEW` dataset.
4. **Upload documentation:** Register a text file as an `ATTACHMENT` dataset.

Each dataset will be linked to the same experimental step.

#### Before you begin — Check your files

**Note:** Instructors will provide the folder 'datasets' containing two files and a zip file. Store the folder named `datasets`, in the same directory as this notebook.

The ZIP archive must be named `Parameter1_Macroscopic.zip`.

Your folder structure should look like this:

```text
WORKSHOP_FOLDER/
│
├── pybis_workshop_exercises.ipynb
│
└── datasets/
    ├── Parameter1_Macroscopic.zip
    ├── Synthetic-image1.png
    └── README-Synthetic-image1.txt
```

#### Your task

1. Run the first code cell to extract the ZIP archive.
2. Upload the extracted files as a `RAW_DATA` dataset.
3. Upload `Synthetic-image1.png` as an `ELN_PREVIEW` dataset.
4. Change the dataset type from `ELN_PREVIEW` to `ATTACHMENT` in the final upload cell.
5. Run the final cell to upload the README file.
6. Open the Data Store GUI and check that all three datasets are attached to `Dummy_Object`.

#### Expected result

The practice Object `Dummy_Object` should contain three datasets:

| Dataset name | Dataset type | Content |
|---|---|---|
| Parameter 1 raw | `RAW_DATA` | Multiple example files extracted from the ZIP archive |
| Synthetic image1 | `ELN_PREVIEW` | An example image |
| README Synthetic image1 | `ATTACHMENT` | A text file explaining the example image |

#### Important notes

- **Only upload datasets to `Dummy_Object`.** Do not modify the original Objects from the GUI tutorial.
- A **Dataset Type** describes the purpose of the uploaded data.
- The `.save()` method registers the dataset and uploads the associated files to openBIS.
- Running an upload cell again may create another dataset containing the same files.
- A ZIP archive is a compressed file that can contain multiple files and folders.

**Think about it:** Why might it be useful to upload a complete folder of experimental files using Python instead of uploading each file manually through the GUI?


In [45]:
# STEP 1: Import the Python libraries we need
# zipfile extracts compressed ZIP archives
# Path helps us work with files and folders

import zipfile
from pathlib import Path

# STEP 2: Define the folder containing our example datasets
# The folder must be in the same directory as the notebook

datasets_folder = Path("datasets")

# Define the ZIP archive we want to extract
zip_path = datasets_folder / "Parameter1_Macroscopic.zip"

# Define where the extracted files will be stored
extract_dir = datasets_folder / "Parameter1_Macroscopic"

# STEP 3: Check that the ZIP archive exists
if not zip_path.is_file():
    raise FileNotFoundError(
        f"File not found: {zip_path}\n"
        "Check that the datasets folder contains the ZIP archive."
    )

# STEP 4: Create the destination folder if necessary
extract_dir.mkdir(parents=True, exist_ok=True)

# STEP 5: Extract the files from the ZIP archive
with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(extract_dir)

# STEP 6: Display a confirmation message
print("ZIP archive extracted successfully!")
print("Extracted files are stored in:", extract_dir)

# Show the extracted files
for file in extract_dir.rglob("*"):
    if file.is_file():
        print("-", file.name)


ZIP archive extracted successfully!
Extracted files are stored in: datasets\Parameter1_Macroscopic
- image_metadata.csv
- Img-1_synthetic_microscopy.png
- Img-2_synthetic_microscopy.png
- Img-3_synthetic_microscopy.png
- Img-4_synthetic_microscopy.png
- Img-5_synthetic_microscopy.png
- README.txt


In [46]:
# STEP 1: Find the Synthesis collection

synthesis_collection = next(col for col in project.get_collections()if col.props.get("$name") == "Synthesis")

# STEP 2: Find our practice Object

Dummy_Object = next(
    object for object in synthesis_collection.get_objects()
    if object.props.get("$name") == "Dummy_Object"
)

# STEP 3: Display the selected Object
# All three datasets will be attached to this Object

print("Collection:", synthesis_collection.props.get("$name"))
print("Practice Object:", Dummy_Object.props.get("$name"))
print("Object identifier:", Dummy_Object.identifier)


Collection: Synthesis
Practice Object: Dummy_Object
Object identifier: /VP.1_NARSHAD/MATERIAL_SYNTHESIS_1234567890/EXP8873


In [47]:
# STEP 1: Prepare a new dataset
# RAW_DATA is the dataset type used for the original research files

raw_dataset = o.new_dataset(
    type="RAW_DATA",

    # Attach the dataset to our practice Object
    object=Dummy_Object.identifier,

    # Upload the files from the extracted folder
    file=str(extract_dir),

    # Add metadata describing the dataset
    props={
        "$name": "Parameter 1 raw",
        "notes": "Synthetic example images for demonstration."
    }
)

# STEP 2: Register the dataset and upload the files
# WARNING: Running this cell again may create another dataset

raw_dataset.save()

# STEP 3: Display the identifier of the new dataset
print("RAW_DATA dataset uploaded!")
print("Dataset identifier:", raw_dataset.permId)

DataSet successfully created.
RAW_DATA dataset uploaded!
Dataset identifier: 20261105162056415-17721



#### Understanding the upload command

The `o.new_dataset()` method prepares a new dataset for registration in openBIS.

The parameters tell pyBIS:

- `type`: Which type of dataset we are uploading.
- `object`: Which Object the dataset belongs to.
- `file`: Which local file or folder should be uploaded.
- `props`: Which metadata should describe the dataset.

The `.save()` method registers the dataset and uploads the files.

**Expected result:** A new `RAW_DATA` dataset is attached to `Dummy_Object`.

**GUI checkpoint:** Open `Dummy_Object` and check whether the dataset and its files are visible.


In [48]:
# STEP 1: Define the image we want to upload

image_path = datasets_folder / "Synthetic-image1.png"

# STEP 2: Check that the image exists
if not image_path.is_file():
    raise FileNotFoundError(f"Image not found: {image_path}")

# STEP 3: Prepare a new image preview dataset
# ELN_PREVIEW is used for preview content in the Lab Notebook

preview_dataset = o.new_dataset(
    type="ELN_PREVIEW",

    # Attach the image to our practice Object
    object=Dummy_Object.identifier,

    # Select the image file
    file=str(image_path),

    # Add descriptive metadata
    props={
        "$name": "Synthetic image1",
        "notes": "Artificial example image for demonstration."
    }
)

# STEP 4: Register and upload the dataset
preview_dataset.save()

# STEP 5: Confirm the upload
print("Image preview uploaded!")
print("Dataset identifier:", preview_dataset.permId)

DataSet successfully created.
Image preview uploaded!
Dataset identifier: 20261105162139749-17722


### YOUR TURN:
**Change the dataset type from "ELN_PREVIEW" to "ATTACHMENT" before running this cell.**

In [49]:
## STEP 1: Define the README file

readme_path = datasets_folder / "README-Synthetic-image1.txt"

# STEP 2: Check that the file exists
if not readme_path.is_file():
    raise FileNotFoundError(f"File not found: {readme_path}")

# STEP 3: Select the correct dataset type
dataset_type = "ATTACHMENT"  # <-- CHANGE THIS TO "ATTACHMENT"

# STEP 4: Check your answer before uploading
# This prevents accidentally registering the wrong dataset type

if dataset_type != "ATTACHMENT":
    raise ValueError(
        "Please change dataset_type to ATTACHMENT before continuing."
    )

# STEP 5: Prepare the dataset

attachment_dataset = o.new_dataset(
    type=dataset_type,

    object=Dummy_Object.identifier,

    file=str(readme_path),

    props={
        "$name": "README Synthetic image1",
        "notes": "README file explaining the synthetic image."
    }
)

# STEP 6: Register and upload the attachment
attachment_dataset.save()

# STEP 7: Confirm the upload
print("README attachment uploaded!")
print("Dataset identifier:", attachment_dataset.permId)


DataSet successfully created.
README attachment uploaded!
Dataset identifier: 20261105162249969-17723




#### GUI checkpoint — Verify your uploaded datasets

Now verify that all three datasets have been uploaded successfully.

1. Open the BAM Data Store Playground in your browser.
2. Navigate to your assigned training Space.
3. Open the `Material_Synthesis` Project.
4. Open the `Synthesis` Collection.
5. Select the practice Object `Dummy_Object`.
6. Find the datasets attached to this Object.

**Expected result:**

You should find three new datasets:

- `Parameter 1 raw` — Type `RAW_DATA`
- `Synthetic image1` — Type `ELN_PREVIEW`
- `README Synthetic image1` — Type `ATTACHMENT`

Open the datasets and check that the expected files are available.

**Congratulations!** You have used pyBIS to upload research data and documentation to the Data Store.

**Think about it:** How could this workflow help you upload and organize data from multiple experiments automatically?




### Challenge 5 — Automate a repetitive task (30 minutes)

#### Why do this?

Imagine your project contains 100 experimental steps, and you need a list of their names, identifiers, and descriptions.

You could open every record individually in the Data Store graphical user interface (GUI) and copy the information into a spreadsheet.

Instead, **Python can collect this information automatically**, saving time and reducing the risk of manual copying errors.

This is particularly useful when working with large numbers of experimental records or preparing reports.

#### In this exercise...

You will use Python to collect metadata from experimental steps in two Collections: `Synthesis` and `Parameter_1`.

You will perform four steps:

1. **Retrieve data:** Find the experimental steps in the selected Collections.
2. **Collect metadata:** Extract the Collection name, Object identifier, Object name, and experimental description.
3. **Create a table:** Organize the information into a table using **pandas**.
4. **Export and read a CSV:** Save the table as a CSV file and open it again using Python.

A **loop** tells Python to repeat the same operation for multiple records.

A **CSV file** (Comma-Separated Values) is a simple text file that stores tabular data. It can be opened in spreadsheet software such as Excel.

#### Your task

1. Run the first code cell to retrieve metadata from the selected Collections.
2. Examine the resulting table.
3. Change the value of `filename` to a filename of your choice, keeping the `.csv` extension.
4. Run the cell to export the table.
5. Find and open the CSV file in VS Code or Excel.
6. Optionally, change a value in the CSV file and save it.
7. Run the second code cell to read the CSV file back into Python.

#### Expected result

- A table containing the metadata of the experimental steps from `Synthesis` and `Parameter_1`.
- A CSV file saved in your notebook's working directory.
- The ability to open and inspect the exported CSV file.
- If you edited and saved the CSV, the modified values appear when Python reads the file again.

#### Important notes

- **This exercise only reads data from openBIS.** It does not modify any research records in the Data Store.
- The exported CSV file is a separate copy of the metadata.
- **Editing the CSV does not update openBIS.** Changes to the CSV remain local unless you explicitly write code to update the records in openBIS.
- Running the export cell again with the same filename will overwrite the existing CSV file.

**Think about it:** How could this approach help you prepare reports for projects containing hundreds or thousands of experimental steps?

**SUGGESTION - consider following code and decide what version to keep to facilitate understanding for beginners, providing code comments**
**Consider - the original code uses next() and enumerate() together. Since enumerate() is not needed, I would remove it.**
**consider to retrieve the project collections only once and separate the collection selection from the data collection loop.**

**Two technical improvements suggested by AI: 

1. make the file location explicit. By default, report.to_csv(filename) saves the file in Python's current working directory, which may differ from the notebook's location in VS Code. To make this easier for beginners, you could display the full output path:

from pathlib import Path

print("CSV saved at:", Path(filename).resolve())

2. make the CSV exercise safe to repeat. If participants edit the CSV and then rerun the export cell, their changes will be overwritten. I would add a warning immediately above the export command:

### WARNING:
### Running this cell again will overwrite the existing CSV file.
report.to_csv(filename, index=False)

In [ ]:
# **Suggestion - Code to review**

# STEP 1: Choose which Collections to include in our report
# We will collect metadata from "Synthesis" and "Parameter 1"

collection_names = ["Synthesis", "Parameter 1"]

# STEP 2: Retrieve the Collections available in our Project

project_collections = project.get_collections()

# STEP 3: Create an empty list to store our report data
# Each Object will become one row in the final table

report_rows = []

# STEP 4: Go through the Collections in our Project
# A "for" loop repeats the instructions for each Collection

for collection in project_collections:

    # Read the human-readable Collection name
    collection_name = collection.props.get("$name")

    # Only process Collections selected for our report
    if collection_name in collection_names:

        # STEP 5: Retrieve all Objects in this Collection

        for obj in collection.get_objects():

            # STEP 6: Extract metadata from each Object
            # Each dictionary represents one row in our report

            report_rows.append({
                "collection": collection_name,
                "identifier": obj.identifier,
                "name": obj.props.get("$name"),
                "description": obj.props.get(
                    "experimental_step.experimental_description"
                ),
            })

# STEP 7: Convert the collected data into a pandas DataFrame
# A DataFrame is a table with rows and columns

report = pd.DataFrame(
    report_rows,
    columns=["collection", "identifier", "name", "description"]
)

# STEP 8: Display the report in the notebook

display(report)

# STEP 9: Choose a filename for the CSV report
# YOUR TURN: Change the filename, but keep the .csv extension

filename = "my_openbis_experimental_steps.csv"

# STEP 10: Export the DataFrame to a CSV file
# index=False prevents pandas from adding an extra index column

report.to_csv(filename, index=False)

# STEP 11: Confirm that the file was created

print(f"Exported {len(report)} records to {filename}")



#### Check — Open your exported CSV file

After running the previous cell, locate the CSV file in your notebook's working directory.

1. Open the file in VS Code or Excel.
2. Check that it contains the four columns: `collection`, `identifier`, `name`, and `description`.
3. Compare the Object names with those displayed in the Data Store GUI.

**Expected result:** The CSV contains the same metadata as the table displayed in your notebook.

**Optional task:** Change one description in the CSV file and save it.

In the next step, we will read the CSV file back into Python to see the saved contents.

**Remember:** Changing the CSV file does not change the metadata stored in openBIS.


In [ ]:
#**Suggested code to review**

# STEP 1: Read the CSV file we created in the previous cell
# pandas uses read_csv() to load the file into a DataFrame

# The variable "filename" already contains the name of our CSV file
# You do not need to enter the filename again

table = pd.read_csv(filename)

# STEP 2: Display the contents of the CSV file
# If you edited and saved the CSV, you should see your changes here

display(table)

# STEP 3: Display the number of rows in the CSV file

print("Number of records in the CSV:", len(table))


In [ ]:
# Choose the collections to include in the report.
#Note what is here meant? participants can edit the name of the .csv file 


selected_collections = [next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Synthesis'), 
                        next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Parameter 1')]

report_rows = []
for collection in selected_collections:
    for obj in collection.get_objects():
        report_rows.append({
            "collection": collection.props.get("$name"),
            "identifier": obj.identifier,
            "name": obj.props.get("$name"),
            "description": obj.props.get("experimental_step.experimental_description"),
        })

report = pd.DataFrame(
    report_rows, columns=["collection", "identifier", "name", "description"]
)

display(report)

filename = "my_openbis_experimental_steps.csv"  # <-- customize
report.to_csv(filename, index=False)

print(f"Exported {len(report)} records to {filename}")


#### Final checkpoint — What have you learned?

You have now used Python to:

- Retrieve experimental steps from multiple Collections.
- Extract metadata from openBIS Objects.
- Organize the information into a table using pandas.
- Export the table to a CSV file.
- Read the CSV file back into Python.

**Congratulations!** You have automated a repetitive research data management task using pyBIS.

**Think about it:**

- What other metadata would be useful to include in your report?
- How could this workflow save time when working with hundreds of experimental records?
- Why is it important to distinguish between a CSV export and the original data stored in openBIS?


**Check:** Open the generated CSV. Does it contain the names you saw in the GUI?

**Optional:** Edit any cells in the CSV file to see the changes reflected in the next code cell.

In [ ]:
# We will now read the generated CSV file called 'my_openbis_experimental_steps.csv' or the name you customized it to.

table = pd.read_csv(filename) # <-- the name of the file is stored in the filename variable so you do not need to change anything here even if you customized 
                                    # the name of the file in the previous cell

display(table)# <-- This will display the contents of the excel file



---
## Step 5 — Wrap-up (20 minutes)
- **Retrieve:** Which pyBIS command gets an existing experimental step?
- **Modify:** What does `.save()` do? Why should you verify in the GUI?
- **Automate:** Which manual GUI task could you replace with a loop and a CSV export?

**Takeaway:** You do not have to memorize Python. You can start with working code, change a few values, and automate a useful openBIS task.

